# Project Phase II: Applied Data Engineering and Machine Learning

In [1]:
import pandas as pd

# Load the raw dataset and create a deep copy for further processing
df = pd.read_csv("../data/student-course-completion-prediction-dataset-trimmed.csv")
df_copy = df.copy(deep=True)

display(df_copy.head())

,Student_ID,Name,Gender,Age,Education_Level,Employment_Status,City,Device_Type,Internet_Connection_Quality,Course_ID,...,Enrollment_Date,Payment_Mode,Fee_Paid,Discount_Used,Payment_Amount,App_Usage_Percentage,Reminder_Emails_Clicked,Support_Tickets_Raised,Satisfaction_Rating,Completed
0,STU100000,Vihaan Patel,Male,19,Diploma,Student,Indore,Laptop,Medium,C102,...,01-06-2024,Scholarship,No,No,1740,49,3,4,3.5,Completed
1,STU100001,Arjun Nair,Female,17,Bachelor,Student,Delhi,Laptop,Low,C106,...,27-04-2025,Credit Card,Yes,No,6147,86,0,0,4.5,Not Completed
2,STU100002,Aditya Bhardwaj,Female,34,Master,Student,Chennai,Mobile,Medium,C101,...,20-01-2024,NetBanking,Yes,No,4280,85,1,0,5.0,Completed
3,STU100003,Krishna Singh,Female,29,Diploma,Employed,Surat,Mobile,High,C105,...,13-05-2025,UPI,Yes,No,3812,42,2,3,3.8,Completed
4,STU100004,Krishna Nair,Female,19,Master,Self-Employed,Lucknow,Laptop,Medium,C106,...,19-12-2024,Debit Card,Yes,Yes,5486,91,3,0,4.0,Completed


In [2]:
for col in df_copy.columns:
    print(f"{col}: {df_copy[col].dtype}, unique: {df_copy[col].nunique(dropna=True)}")

Student_ID: str, unique: 30000
Name: str, unique: 300
Gender: str, unique: 3
Age: int64, unique: 33
Education_Level: str, unique: 5
Employment_Status: str, unique: 4
City: str, unique: 15
Device_Type: str, unique: 3
Internet_Connection_Quality: str, unique: 3
Course_ID: str, unique: 8
Course_Name: str, unique: 8
Category: str, unique: 5
Course_Level: str, unique: 3
Course_Duration_Days: int64, unique: 8
Instructor_Rating: float64, unique: 7
Login_Frequency: int64, unique: 16
Average_Session_Duration_Min: int64, unique: 72
Video_Completion_Rate: float64, unique: 929
Discussion_Participation: int64, unique: 13
Time_Spent_Hours: float64, unique: 209
Days_Since_Last_Login: int64, unique: 71
Notifications_Checked: int64, unique: 19
Peer_Interaction_Score: float64, unique: 101
Assignments_Submitted: int64, unique: 11
Assignments_Missed: int64, unique: 11
Quiz_Attempts: int64, unique: 16
Quiz_Score_Avg: float64, unique: 668
Project_Grade: float64, unique: 811
Progress_Percentage: float64, uni

### Data Preprocessing and Feature Engineering

In [3]:
# Check and correct incorrect data types
print("=== Normalizing Data Types ===\n")

integer_columns = [
    "Age",
    "Course_Duration_Days",
    "Login_Frequency",
    "Average_Session_Duration_Min",
    "Discussion_Participation",
    "Days_Since_Last_Login",
    "Notifications_Checked",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Rewatch_Count",
    "Payment_Amount",
    "App_Usage_Percentage",
    "Reminder_Emails_Clicked",
    "Support_Tickets_Raised",
]
float_columns = [
    "Instructor_Rating",
    "Video_Completion_Rate",
    "Time_Spent_Hours",
    "Peer_Interaction_Score",
    "Quiz_Score_Avg",
    "Project_Grade",
    "Progress_Percentage",
    "Satisfaction_Rating",
    "Attendance_Rate",
]
expected_numeric_dtypes = {
    **dict.fromkeys(integer_columns, "int64"),
    **dict.fromkeys(float_columns, "float64"),
}

for column_name, expected_dtype in expected_numeric_dtypes.items():
    if column_name not in df_copy.columns:
        continue

    original_values = df_copy[column_name]
    converted_values = pd.to_numeric(original_values, errors="coerce")
    invalid_mask = original_values.notna() & converted_values.isna()

    if invalid_mask.any():
        examples = original_values[invalid_mask].head(5).tolist()
        print(
            f"{column_name}: {invalid_mask.sum()} values could not be converted "
            f"to {expected_dtype}; examples: {examples}"
        )

    fractional_mask = converted_values.notna() & converted_values.mod(1).ne(0)
    if expected_dtype == "int64" and fractional_mask.any():
        print(
            f"{column_name}: {fractional_mask.sum()} fractional values found; "
            "preserving the column as float64"
        )
        df_copy[column_name] = converted_values.astype("float64")
    elif expected_dtype == "int64" and not converted_values.isna().any():
        df_copy[column_name] = converted_values.astype("int64")
    else:
        df_copy[column_name] = converted_values.astype("float64")

# Dataset dates use day-month-year format.
if "Enrollment_Date" in df_copy.columns:
    original_dates = df_copy["Enrollment_Date"]
    parsed_dates = pd.to_datetime(
        original_dates, format="%d-%m-%Y", errors="coerce"
    )
    invalid_dates = original_dates.notna() & parsed_dates.isna()

    if invalid_dates.any():
        examples = original_dates[invalid_dates].head(5).tolist()
        print(
            f"Enrollment_Date: {invalid_dates.sum()} values could not be parsed; "
            f"examples: {examples}"
        )

    df_copy["Enrollment_Date"] = parsed_dates

print("\nResulting data types:")
print(df_copy.dtypes.to_string())

=== Normalizing Data Types ===


Resulting data types:
Student_ID                                 str
Name                                       str
Gender                                     str
Age                                      int64
Education_Level                            str
Employment_Status                          str
City                                       str
Device_Type                                str
Internet_Connection_Quality                str
Course_ID                                  str
Course_Name                                str
Category                                   str
Course_Level                               str
Course_Duration_Days                     int64
Instructor_Rating                      float64
Login_Frequency                          int64
Average_Session_Duration_Min             int64
Video_Completion_Rate                  float64
Discussion_Participation                 int64
Time_Spent_Hours                       float64
Days_

In [4]:
# Address missing values and duplicate records

# Display duplicate Student_ID records
duplicate_student_rows = df_copy[df_copy["Student_ID"].duplicated(keep=False)]

print("Duplicate records with the same Student_ID:")
if len(duplicate_student_rows) > 0:
    print(duplicate_student_rows.to_string(index=False))
else:
    print("No duplicate Student_ID records found.")

# Display completely identical duplicate records
complete_duplicate_rows = df_copy[df_copy.duplicated(keep=False)]

print("\nComplete duplicate records:")
if len(complete_duplicate_rows) > 0:
    print(complete_duplicate_rows.to_string(index=False))
else:
    print("No complete duplicate records found.")

# Check columns with missing values
missing = df_copy.isna().sum()
missing_columns = missing[missing > 0].sort_values(ascending=False)

print("\nColumns containing missing values:")

if missing_columns.empty:
    print("No missing values found.")
else:
    for column_name, missing_count in missing_columns.items():
        print(f"{column_name}: {missing_count}")

    # Display rows containing missing values
    rows_with_missing = df_copy[df_copy.isna().any(axis=1)]

    print("\nRows containing missing values:")
    print(rows_with_missing.to_string(index=False))

print("\nDataset shape after inspection:")
print(f"Rows: {df_copy.shape[0]}, Columns: {df_copy.shape[1]}")

Duplicate records with the same Student_ID:
No duplicate Student_ID records found.

Complete duplicate records:
No complete duplicate records found.

Columns containing missing values:
No missing values found.

Dataset shape after inspection:
Rows: 30000, Columns: 40


In [5]:
# Inspect and fix invalid values

print("=== Inspecting Invalid Values ===\n")

# Check for negative values in columns that should be non-negative
numeric_cols = df_copy.select_dtypes(include="number").columns

invalid_values_found = False

for col in numeric_cols:
    negative_count = (df_copy[col] < 0).sum()

    if negative_count > 0:
        invalid_values_found = True
        print(f"{col}: {negative_count} negative values found")

        # Show examples of negative values
        negative_examples = df_copy[df_copy[col] < 0][col].head(5).tolist()
        print(f"  Examples: {negative_examples}")

# Check for out-of-range values in percentage columns
if "Attendance_Rate" in df_copy.columns:
    out_of_range = (
        (df_copy["Attendance_Rate"] < 0) | (df_copy["Attendance_Rate"] > 100)
    ).sum()
    if out_of_range > 0:
        invalid_values_found = True
        print(f"Attendance_Rate: {out_of_range} values outside 0-100 range")

if not invalid_values_found:
    print("No invalid numeric values detected.\n")
else:
    print()

# Check for unexpected values in expected binary/categorical columns
binary_cols = {
    "Fee_Paid": ["Yes", "No"],
    "Discount_Used": ["Yes", "No"],
    "Completed": ["Completed", "Not Completed"],
}

print("=== Inspecting Expected Categorical Values ===\n")

categorical_anomalies = False

for col, expected_values in binary_cols.items():
    if col in df_copy.columns:
        actual_values = df_copy[col].unique()
        unexpected = [
            value for value in actual_values if value not in expected_values and pd.notna(value)
        ]

        if unexpected:
            categorical_anomalies = True
            print(f"{col}: Unexpected values found")
            print(f"  Expected: {expected_values}")
            print(f"  Found: {list(set(unexpected))}")

if not categorical_anomalies:
    print("All categorical columns have expected values.\n")
else:
    print()

# Check for inconsistent categories with case/whitespace variations
print("=== Inspecting Inconsistent Categories ===\n")

category_check_cols = [
    "Gender",
    "Education_Level",
    "Employment_Status",
    "City",
    "Device_Type",
    "Internet_Connection_Quality",
    "Course_ID",
    "Course_Name",
    "Category",
    "Course_Level",
    "Payment_Mode",
    "Fee_Paid",
    "Discount_Used",
    "Completed",
]

inconsistency_found = False

for col in category_check_cols:
    if col in df_copy.columns:
        original_count = df_copy[col].nunique(dropna=False)

        cleaned_count = (
            df_copy[col].astype("string").str.strip().str.lower().nunique(dropna=False)
        )

        if original_count != cleaned_count:
            inconsistency_found = True
            print(f"{col}: {original_count} original vs {cleaned_count} standardised")

            # Show examples of variations
            original_values = df_copy[col].dropna().unique()[:10]
            print(f"  Sample values: {list(original_values)}")

if not inconsistency_found:
    print("No categorical inconsistencies detected.\n")
else:
    print()

=== Inspecting Invalid Values ===

No invalid numeric values detected.

=== Inspecting Expected Categorical Values ===

All categorical columns have expected values.

=== Inspecting Inconsistent Categories ===

No categorical inconsistencies detected.



In [6]:
# Split before any data-dependent preprocessing. This is to avoid data leakage during preprocessing.
from sklearn.model_selection import train_test_split

X_all = df_copy.drop(columns=["Completed"])
y_all = df_copy["Completed"].map({"Not Completed": 0, "Completed": 1})

if y_all.isna().any():
    raise ValueError("Completed contains labels outside the expected categories.")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all,
)

X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()
initial_train_rows = len(X_train_raw)

print("=== Leakage-Safe Train/Test Split ===")
print(f"Training rows: {len(X_train_raw)}")
print(f"Test rows: {len(X_test_raw)}")
print("Target proportions:")
print("  Train:")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print("  Test:")
print(y_test.value_counts(normalize=True).sort_index().round(3))

=== Leakage-Safe Train/Test Split ===
Training rows: 24000
Test rows: 6000
Target proportions:
  Train:
Completed
0    0.507
1    0.493
Name: proportion, dtype: float64
  Test:
Completed
0    0.507
1    0.493
Name: proportion, dtype: float64


In [7]:
# Investigate outliers using training data only
import numpy as np

print("=== Investigating Training-Set Outliers Using IQR Method ===\n")

numeric_cols = X_train_raw.select_dtypes(include="number").columns

valid_ranges = {
    "Age": (17, 60),
    "Instructor_Rating": (0, 5),
    "Satisfaction_Rating": (0, 5),
    "Video_Completion_Rate": (0, 100),
    "Progress_Percentage": (0, 100),
    "App_Usage_Percentage": (0, 100),
    "Quiz_Score_Avg": (0, 100),
    "Project_Grade": (0, 100),
    "Course_Duration_Days": (0, 90),
    "Average_Session_Duration_Min": (0, 80),
    "Time_Spent_Hours": (0, 30),
    "Days_Since_Last_Login": (0, 100),
    "Payment_Amount": (0, None),
}

outlier_summary = {}

for col in numeric_cols:
    training_values = X_train_raw[col]
    q1 = training_values.quantile(0.25)
    q3 = training_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (training_values < lower_bound) | (training_values > upper_bound)
    outlier_count = outlier_mask.sum()

    range_violations = 0
    if col in valid_ranges:
        min_valid, max_valid = valid_ranges[col]
        if max_valid is None:
            range_mask = training_values < min_valid
        else:
            range_mask = (training_values < min_valid) | (training_values > max_valid)
        range_violations = range_mask.sum()

    outlier_summary[col] = {
        "outlier_count_iqr": outlier_count,
        "outlier_pct": round((outlier_count / len(X_train_raw)) * 100, 2),
        "range_violations": range_violations,
        "min": training_values.min(),
        "max": training_values.max(),
        "mean": training_values.mean(),
        "median": training_values.median(),
        "std": training_values.std(),
        "iqr_bounds": (lower_bound, upper_bound),
    }

print("Training columns with outliers (IQR method > 0.5%):\n")
for col, stats in outlier_summary.items():
    if stats["outlier_pct"] > 0.5 or stats["range_violations"] > 0:
        print(f"{col}:")
        print(
            f"  Min: {stats['min']:.2f}, Max: {stats['max']:.2f}, "
            f"Mean: {stats['mean']:.2f}, Median: {stats['median']:.2f}"
        )
        print(f"  Std Dev: {stats['std']:.2f}")
        print(f"  IQR Outliers: {stats['outlier_count_iqr']} ({stats['outlier_pct']}%)")
        if stats["range_violations"] > 0:
            print(f"  Range Violations: {stats['range_violations']}")
        print(
            f"  IQR Bounds: ({stats['iqr_bounds'][0]:.2f}, "
            f"{stats['iqr_bounds'][1]:.2f})\n"
        )

print("\n=== Identifying Training Records with Range Violations ===\n")
violation_mask = pd.Series(False, index=X_train_raw.index)

for col, (min_val, max_val) in valid_ranges.items():
    if col in X_train_raw.columns:
        if max_val is None:
            col_violation = X_train_raw[col] < min_val
        else:
            col_violation = (X_train_raw[col] < min_val) | (X_train_raw[col] > max_val)
        violation_mask |= col_violation

records_with_violations = X_train_raw.loc[violation_mask]
print(f"Training records with range violations: {len(records_with_violations)}")
if len(X_train_raw) > 0:
    print(
        "Percentage of training set: "
        f"{len(records_with_violations) / len(X_train_raw) * 100:.2f}%\n"
    )

if not records_with_violations.empty:
    print("Sample training records with violations:")
    violation_cols = ["Student_ID"] + list(valid_ranges.keys())
    violation_cols = [col for col in violation_cols if col in records_with_violations.columns]
    print(records_with_violations[violation_cols].head(10).to_string(index=False))

=== Investigating Training-Set Outliers Using IQR Method ===

Training columns with outliers (IQR method > 0.5%):

Average_Session_Duration_Min:
  Min: 5.00, Max: 76.00, Mean: 33.86, Median: 34.00
  Std Dev: 10.37
  IQR Outliers: 137 (0.57%)
  IQR Bounds: (6.00, 62.00)

Discussion_Participation:
  Min: 0.00, Max: 12.00, Mean: 2.32, Median: 2.00
  Std Dev: 1.58
  IQR Outliers: 307 (1.28%)
  IQR Bounds: (-2.00, 6.00)

Time_Spent_Hours:
  Min: 0.50, Max: 25.60, Mean: 3.89, Median: 2.70
  Std Dev: 3.80
  IQR Outliers: 238 (0.99%)
  IQR Bounds: (-8.20, 15.00)

Days_Since_Last_Login:
  Min: 0.00, Max: 99.00, Mean: 6.13, Median: 4.00
  Std Dev: 7.02
  IQR Outliers: 937 (3.9%)
  IQR Bounds: (-11.00, 21.00)

Notifications_Checked:
  Min: 0.00, Max: 18.00, Mean: 5.22, Median: 5.00
  Std Dev: 2.39
  IQR Outliers: 256 (1.07%)
  IQR Bounds: (-0.50, 11.50)

Quiz_Attempts:
  Min: 0.00, Max: 16.00, Mean: 3.77, Median: 4.00
  Std Dev: 2.02
  IQR Outliers: 197 (0.82%)
  IQR Bounds: (-2.50, 9.50)

Rewatc

In [8]:
# Handle outliers without learning from the test set

print("=== Handling Outliers ===\n")

# Apply only predefined domain limits to both splits
percentage_cols = [
    "Video_Completion_Rate",
    "Progress_Percentage",
    "App_Usage_Percentage",
    "Quiz_Score_Avg",
    "Project_Grade",
]

for col in percentage_cols:
    if col in X_train_raw.columns:
        for split_name, split_data in (("train", X_train_raw), ("test", X_test_raw)):
            before_count = ((split_data[col] < 0) | (split_data[col] > 100)).sum()
            split_data[col] = split_data[col].clip(0, 100)
            if before_count > 0:
                print(f"{split_name} {col}: capped {before_count} values to [0, 100]")

rating_cols = ["Instructor_Rating", "Satisfaction_Rating"]
for col in rating_cols:
    if col in X_train_raw.columns:
        for split_name, split_data in (("train", X_train_raw), ("test", X_test_raw)):
            before_count = ((split_data[col] < 0) | (split_data[col] > 5)).sum()
            split_data[col] = split_data[col].clip(0, 5)
            if before_count > 0:
                print(f"{split_name} {col}: capped {before_count} values to [0, 5]")

if "Age" in X_train_raw.columns:
    for split_name, split_data in (("train", X_train_raw), ("test", X_test_raw)):
        before_count = ((split_data["Age"] < 17) | (split_data["Age"] > 80)).sum()
        split_data["Age"] = split_data["Age"].clip(17, 80)
        if before_count > 0:
            print(f"{split_name} Age: capped {before_count} values to [17, 80]")

print("\nRemoving extreme IQR outliers from training data only (k=3):\n")
rows_to_remove = []

for col in numeric_cols:
    training_values = X_train_raw[col]
    q1 = training_values.quantile(0.25)
    q3 = training_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 3 * iqr
    upper_bound = q3 + 3 * iqr
    extreme_mask = (training_values < lower_bound) | (training_values > upper_bound)

    if extreme_mask.any():
        print(f"{col}: {extreme_mask.sum()} training outliers; bounds [{lower_bound:.2f}, {upper_bound:.2f}]")
        rows_to_remove.extend(X_train_raw.index[extreme_mask].tolist())

rows_to_remove = sorted(set(rows_to_remove))
if rows_to_remove:
    X_train_raw = X_train_raw.drop(index=rows_to_remove)
    y_train = y_train.drop(index=rows_to_remove)
    print(f"Removed {len(rows_to_remove)} training rows; test rows remain unchanged.")
else:
    print("No extreme training outliers detected.")

print(f"\nTraining rows retained: {len(X_train_raw)} of {initial_train_rows}")
print(f"Test rows retained: {len(X_test_raw)}")

=== Handling Outliers ===


Removing extreme IQR outliers from training data only (k=3):

Discussion_Participation: 9 training outliers; bounds [-5.00, 9.00]
Time_Spent_Hours: 3 training outliers; bounds [-16.90, 23.70]
Days_Since_Last_Login: 208 training outliers; bounds [-23.00, 33.00]
Notifications_Checked: 5 training outliers; bounds [-5.00, 16.00]
Quiz_Attempts: 2 training outliers; bounds [-7.00, 14.00]
Rewatch_Count: 10 training outliers; bounds [-5.00, 9.00]
Reminder_Emails_Clicked: 10 training outliers; bounds [-5.00, 9.00]
Support_Tickets_Raised: 57 training outliers; bounds [-3.00, 4.00]
Removed 302 training rows; test rows remain unchanged.

Training rows retained: 23698 of 24000
Test rows retained: 6000


In [9]:
# Final summary after training-only outlier handling
print("=== Data Quality Summary ===\n")
print(f"Rows in source dataframe: {len(df_copy)}")
print(f"Training rows before IQR removal: {initial_train_rows}")
print(f"Training rows removed by training-only IQR rule: {initial_train_rows - len(X_train_raw)}")
print(f"Training rows retained: {len(X_train_raw)}")
print(f"Test rows retained unchanged: {len(X_test_raw)}\n")

print("Training Numeric Columns Statistics:")
print("-" * 80)
numeric_summary = X_train_raw.select_dtypes(include="number").describe().T.round(2)
print(numeric_summary.to_string())
print("\n" + "=" * 80)
print("Preprocessing statistics are derived from training data only.")

=== Data Quality Summary ===

Rows in source dataframe: 30000
Training rows before IQR removal: 24000
Training rows removed by training-only IQR rule: 302
Training rows retained: 23698
Test rows retained unchanged: 6000

Training Numeric Columns Statistics:
--------------------------------------------------------------------------------
                                count     mean      std   min      25%      50%     75%     max
Age                           23698.0    25.71     5.60  17.0    21.00    26.00    30.0    49.0
Course_Duration_Days          23698.0    51.75    20.31  25.0    30.00    45.00    60.0    90.0
Instructor_Rating             23698.0     4.44     0.20   4.1     4.30     4.50     4.6     4.7
Login_Frequency               23698.0     4.78     1.84   0.0     3.00     5.00     6.0    15.0
Average_Session_Duration_Min  23698.0    33.88    10.37   5.0    27.00    34.00    41.0    76.0
Video_Completion_Rate         23698.0    62.32    19.51   5.0    48.60    64.15    77

In [10]:
# Encode categorical features using training data only
from sklearn.preprocessing import OrdinalEncoder

print("=== Leakage-Safe Categorical Encoding ===\n")

identifier_cols = ["Student_ID", "Name", "Course_Name", "Enrollment_Date"]
X_train_encoded = X_train_raw.drop(columns=identifier_cols, errors="ignore").copy()
X_test_encoded = X_test_raw.drop(columns=identifier_cols, errors="ignore").copy()

categorical_features = [
    "Gender",
    "Education_Level",
    "Employment_Status",
    "City",
    "Device_Type",
    "Internet_Connection_Quality",
    "Course_ID",
    "Category",
    "Payment_Mode",
    "Fee_Paid",
    "Discount_Used",
]
categorical_features = [
    col for col in categorical_features if col in X_train_encoded.columns
]

for col in categorical_features:
    X_train_encoded[col] = X_train_encoded[col].astype("string").fillna("__MISSING__")
    X_test_encoded[col] = X_test_encoded[col].astype("string").fillna("__MISSING__")

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
if categorical_features:
    X_train_encoded[categorical_features] = categorical_encoder.fit_transform(
        X_train_encoded[categorical_features]
    )
    X_test_encoded[categorical_features] = categorical_encoder.transform(
        X_test_encoded[categorical_features]
    )

# Preserve the known course-level order rather than deriving it from all rows.
course_level_order = {"Beginner": 0, "Intermediate": 1, "Advanced": 2}
if "Course_Level" in X_train_encoded.columns:
    for split_name, split_data in (("train", X_train_encoded), ("test", X_test_encoded)):
        encoded_levels = split_data["Course_Level"].map(course_level_order)
        unknown_levels = split_data["Course_Level"].notna() & encoded_levels.isna()
        if unknown_levels.any():
            print(
                f"{split_name}: {unknown_levels.sum()} unrecognized Course_Level values "
                "encoded as -1"
            )
        split_data["Course_Level"] = encoded_levels.fillna(-1).astype("int64")

headers = X_train_encoded.columns.tolist()
if headers != X_test_encoded.columns.tolist():
    raise ValueError("Training and test feature columns do not match.")

print(f"Encoded training shape: {X_train_encoded.shape}")
print(f"Encoded test shape: {X_test_encoded.shape}")
print(f"Encoded categorical features: {categorical_features}")
print("Unseen test categories are encoded as -1.")

=== Leakage-Safe Categorical Encoding ===

Encoded training shape: (23698, 35)
Encoded test shape: (6000, 35)
Encoded categorical features: ['Gender', 'Education_Level', 'Employment_Status', 'City', 'Device_Type', 'Internet_Connection_Quality', 'Course_ID', 'Category', 'Payment_Mode', 'Fee_Paid', 'Discount_Used']
Unseen test categories are encoded as -1.


In [11]:
# Scale numerical variables using training data only
from sklearn.preprocessing import StandardScaler

print("=== Leakage-Safe Numerical Scaling ===\n")

continuous_features = [
    "Age",
    "Course_Duration_Days",
    "Instructor_Rating",
    "Login_Frequency",
    "Average_Session_Duration_Min",
    "Video_Completion_Rate",
    "Discussion_Participation",
    "Time_Spent_Hours",
    "Days_Since_Last_Login",
    "Notifications_Checked",
    "Peer_Interaction_Score",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Quiz_Score_Avg",
    "Project_Grade",
    "Progress_Percentage",
    "Rewatch_Count",
    "Payment_Amount",
    "App_Usage_Percentage",
    "Reminder_Emails_Clicked",
    "Support_Tickets_Raised",
    "Satisfaction_Rating",
]
continuous_features = [col for col in continuous_features if col in headers]

X_train_scaled_df = X_train_encoded.copy()
X_test_scaled_df = X_test_encoded.copy()
scaler = StandardScaler()

if continuous_features:
    X_train_scaled_df[continuous_features] = scaler.fit_transform(
        X_train_encoded[continuous_features]
    )
    X_test_scaled_df[continuous_features] = scaler.transform(
        X_test_encoded[continuous_features]
    )

print(f"Scaled {len(continuous_features)} continuous features.")
print(f"Training shape: {X_train_scaled_df.shape}")
print(f"Test shape: {X_test_scaled_df.shape}")
if continuous_features:
    print(
        "Mean of scaled training features: "
        f"{X_train_scaled_df[continuous_features].mean().mean():.6f}"
    )
print("Scaler was fitted on training data only.")

=== Leakage-Safe Numerical Scaling ===

Scaled 23 continuous features.
Training shape: (23698, 35)
Test shape: (6000, 35)
Mean of scaled training features: -0.000000
Scaler was fitted on training data only.


In [12]:
# Discretise continuous numerical variables using training data only
from sklearn.preprocessing import KBinsDiscretizer

print("=== Leakage-Safe Quantile Discretisation ===\n")

print("Strategy: Quantile binning with up to 5 bins, fitted on training data only.")
discretise_features = [
    "Age",
    "Course_Duration_Days",
    "Average_Session_Duration_Min",
    "Video_Completion_Rate",
    "Quiz_Score_Avg",
    "Project_Grade",
    "Progress_Percentage",
    "Payment_Amount",
]
discretise_features = [col for col in discretise_features if col in headers]

X_train_discretised = X_train_scaled_df.copy()
X_test_discretised = X_test_scaled_df.copy()
discretiser = KBinsDiscretizer(
    n_bins=5,
    encode="ordinal",
    strategy="quantile",
)

if discretise_features:
    X_train_discretised[discretise_features] = discretiser.fit_transform(
        X_train_scaled_df[discretise_features]
    )
    X_test_discretised[discretise_features] = discretiser.transform(
        X_test_scaled_df[discretise_features]
    )

print("Discretisation results:")
for feature in discretise_features:
    train_bins = sorted(X_train_discretised[feature].unique())
    test_bins = sorted(X_test_discretised[feature].unique())
    print(f"  {feature}: train bins {train_bins}; test bins {test_bins}")

X_train_final = X_train_discretised.copy()
X_test_final = X_test_discretised.copy()
y_train_final = y_train.copy()
y_test_final = y_test.copy()

print(f"\nTraining features: {X_train_final.shape}")
print(f"Test features: {X_test_final.shape}")
print("Discretizer was fitted on training data only.")

=== Leakage-Safe Quantile Discretisation ===

Strategy: Quantile binning with up to 5 bins, fitted on training data only.
Discretisation results:
  Age: train bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]; test bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  Course_Duration_Days: train bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]; test bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  Average_Session_Duration_Min: train bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]; test bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  Video_Completion_Rate: train bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]; test bins [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.floa

In [13]:
# Assess class balance using training labels only for the decision
class_counts_train = y_train_final.value_counts().sort_index()
class_counts_test = y_test_final.value_counts().sort_index()

if len(class_counts_train) < 2 or (class_counts_train == 0).any():
    raise ValueError("Training data must contain at least two classes to assess imbalance.")

class_distribution = pd.DataFrame(
    {
        "Train count": class_counts_train,
        "Test count": class_counts_test,
    }
).fillna(0).astype(int)
class_distribution["Train percent"] = (
    class_distribution["Train count"] / len(y_train_final) * 100
).round(2)
class_distribution["Test percent"] = (
    class_distribution["Test count"] / len(y_test_final) * 100
).round(2)

display(class_distribution)

imbalance_ratio = class_counts_train.max() / class_counts_train.min()
imbalance_threshold = 1.5
needs_resampling = imbalance_ratio >= imbalance_threshold

print(f"Training majority/minority ratio: {imbalance_ratio:.3f}:1")
print(f"Resampling threshold: {imbalance_threshold:.1f}:1")
print(f"Resampling needed: {needs_resampling}")
print("Test labels are reported for reference only and do not determine this decision.")

,Train count,Test count,Train percent,Test percent
Completed,,,,
0,11988,3042,50.59,50.7
1,11710,2958,49.41,49.3


Training majority/minority ratio: 1.024:1
Resampling threshold: 1.5:1
Resampling needed: False
Test labels are reported for reference only and do not determine this decision.


In [14]:
# Resample training data only when imbalance is substantial
from sklearn.utils import resample

X_train_balanced = X_train_final.reset_index(drop=True).copy()
y_train_balanced = y_train_final.reset_index(drop=True).copy()

if needs_resampling:
    majority_class = class_counts_train.idxmax()
    minority_class = class_counts_train.idxmin()
    minority_mask = y_train_balanced == minority_class

    X_minority = X_train_balanced.loc[minority_mask]
    y_minority = y_train_balanced.loc[minority_mask]
    X_minority_upsampled, y_minority_upsampled = resample(
        X_minority,
        y_minority,
        replace=True,
        n_samples=int(class_counts_train[majority_class]),
        random_state=42,
    )

    X_majority = X_train_balanced.loc[~minority_mask]
    y_majority = y_train_balanced.loc[~minority_mask]
    X_train_balanced = pd.concat(
        [X_majority, X_minority_upsampled], ignore_index=True
    )
    y_train_balanced = pd.concat(
        [y_majority, y_minority_upsampled], ignore_index=True
    )

    shuffle_order = np.random.default_rng(42).permutation(len(y_train_balanced))
    X_train_balanced = X_train_balanced.iloc[shuffle_order].reset_index(drop=True)
    y_train_balanced = y_train_balanced.iloc[shuffle_order].reset_index(drop=True)
    print("Minority class oversampled in training data.")
else:
    print("Classes are sufficiently balanced; no resampling applied.")

print("Training counts for modeling:")
print(y_train_balanced.value_counts().sort_index())
print(f"Test rows unchanged: {len(X_test_final)}")
print("Use X_train_balanced/y_train_balanced for training and the original test set for evaluation.")

Classes are sufficiently balanced; no resampling applied.
Training counts for modeling:
Completed
0    11988
1    11710
Name: count, dtype: int64
Test rows unchanged: 6000
Use X_train_balanced/y_train_balanced for training and the original test set for evaluation.


In [15]:
# Select a candidate feature subset using training data only
from functools import partial
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Mark encoded categories and discretised values as discrete for mutual information.
discrete_feature_mask = [
    column_name in categorical_features
    or column_name == "Course_Level"
    or column_name in discretise_features
    for column_name in X_train_final.columns
]
mutual_info_score = partial(
    mutual_info_classif,
    discrete_features=discrete_feature_mask,
    random_state=42,
)

n_selected_features = min(20, X_train_final.shape[1])
feature_selector = SelectKBest(
    score_func=mutual_info_score,
    k=n_selected_features,
)
feature_selector.fit(X_train_final, y_train_final)

selected_feature_names = X_train_final.columns[feature_selector.get_support()].tolist()
selected_feature_scores = pd.DataFrame(
    {
        "Feature": X_train_final.columns,
        "Mutual Information": feature_selector.scores_,
    }
).sort_values("Mutual Information", ascending=False)

X_train_selected = pd.DataFrame(
    feature_selector.transform(X_train_balanced),
    columns=selected_feature_names,
    index=X_train_balanced.index,
)
X_test_selected = pd.DataFrame(
    feature_selector.transform(X_test_final),
    columns=selected_feature_names,
    index=X_test_final.index,
)

print(f"Selected {len(selected_feature_names)} of {X_train_final.shape[1]} features.")
print(f"Selected training shape: {X_train_selected.shape}")
print(f"Selected test shape: {X_test_selected.shape}")
print("Selected feature names:")
print(selected_feature_names)
print("Selector was fitted on training data only; full-feature datasets remain available.")
display(selected_feature_scores.head(n_selected_features))

Selected 20 of 35 features.
Selected training shape: (23698, 20)
Selected test shape: (6000, 20)
Selected feature names:
['Age', 'Education_Level', 'Employment_Status', 'Internet_Connection_Quality', 'Instructor_Rating', 'Login_Frequency', 'Average_Session_Duration_Min', 'Video_Completion_Rate', 'Discussion_Participation', 'Time_Spent_Hours', 'Assignments_Submitted', 'Assignments_Missed', 'Quiz_Score_Avg', 'Project_Grade', 'Progress_Percentage', 'Payment_Mode', 'Fee_Paid', 'Payment_Amount', 'Reminder_Emails_Clicked', 'Support_Tickets_Raised']
Selector was fitted on training data only; full-feature datasets remain available.


,Feature,Mutual Information
25,Progress_Percentage,0.020775
14,Video_Completion_Rate,0.014341
20,Assignments_Submitted,0.009605
21,Assignments_Missed,0.008472
11,Instructor_Rating,0.006352
27,Payment_Mode,0.004356
12,Login_Frequency,0.004081
28,Fee_Paid,0.003958
30,Payment_Amount,0.003669
23,Quiz_Score_Avg,0.003386


In [16]:
# PCA candidate branch using scaled continuous features only
from sklearn.decomposition import PCA

pca_features = [
    column_name
    for column_name in continuous_features
    if column_name in X_train_scaled_df.columns
]
non_pca_features = [
    column_name
    for column_name in X_train_scaled_df.columns
    if column_name not in pca_features
]

if not pca_features:
    raise ValueError("No continuous features are available for PCA.")

pca_model = PCA(n_components=0.95, svd_solver="full")
train_components = pca_model.fit_transform(
    X_train_scaled_df[pca_features]
)
test_components = pca_model.transform(
    X_test_scaled_df[pca_features]
)

component_names = [
    f"PC{component_number + 1}"
    for component_number in range(pca_model.n_components_)
]
train_pca_df = pd.DataFrame(
    train_components,
    columns=component_names,
    index=X_train_scaled_df.index,
)
test_pca_df = pd.DataFrame(
    test_components,
    columns=component_names,
    index=X_test_scaled_df.index,
)

X_train_pca = pd.concat(
    [X_train_scaled_df[non_pca_features], train_pca_df], axis=1
)
X_test_pca = pd.concat(
    [X_test_scaled_df[non_pca_features], test_pca_df], axis=1
)

print(f"PCA components retained: {pca_model.n_components_}")
print(
    "Continuous-feature variance retained: "
    f"{pca_model.explained_variance_ratio_.sum():.3f}"
)
print(f"PCA training shape: {X_train_pca.shape}")
print(f"PCA test shape: {X_test_pca.shape}")
print("Features after PCA:")
print(X_train_pca.columns.tolist())
print("PCA was fitted on training data only; encoded categorical features were retained.")

PCA components retained: 19
Continuous-feature variance retained: 0.952
PCA training shape: (23698, 31)
PCA test shape: (6000, 31)
Features after PCA:
['Gender', 'Education_Level', 'Employment_Status', 'City', 'Device_Type', 'Internet_Connection_Quality', 'Course_ID', 'Category', 'Course_Level', 'Payment_Mode', 'Fee_Paid', 'Discount_Used', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19']
PCA was fitted on training data only; encoded categorical features were retained.


### Improved LCS-Based System